In [84]:
import sys
sys.path.append("/home/user/문서/workspace/python/src")

from data_load_save import *


In [85]:
csv_file = f"/home/user/문서/workspace/python/data/콩_무역_2023.csv"

df = pd.read_csv(csv_file)

df.head()

,importer,exporter,value
0,"China, mainland",Brazil,71592496.82
1,"China, mainland",United States of America,26506551.32
2,Argentina,Paraguay,5829120.00
3,Argentina,Brazil,4081112.10
4,Thailand,Brazil,2750180.24


In [86]:
import pandas as pd
import networkx as nx
import itertools
import community  # python-louvain (import community.community_louvain as community_louvain 라고 쓰기도 함)


# 1) 데이터 불러와서 방향 + 가중 네트워크 만들기
def build_trade_network(csv_path, src_col="exporter", dst_col="importer", w_col="value"):
    df = pd.read_csv(csv_path)
    # 0 이상인 값만 사용 (필요시 필터)
    df = df[df[w_col] > 0].copy()
    
    G = nx.DiGraph()
    for _, row in df.iterrows():
        src = row[src_col]
        dst = row[dst_col]
        w = float(row[w_col])
        # 이미 간선 있으면 weight 누적
        if G.has_edge(src, dst):
            G[src][dst]["weight"] += w
        else:
            G.add_edge(src, dst, weight=w)
    return G


# 2) 기본 네트워크 특성 계산
def compute_basic_measures(G):
    print("=== Basic topology ===")
    N = G.number_of_nodes()
    M = G.number_of_edges()
    print(f"Nodes (N): {N}")
    print(f"Edges (M): {M}")
    
    # Network density (directional)
    density = nx.density(G)
    print(f"Density ρ: {density:.4f}")
    
    # 평균 경로 길이 & 직경: 약하게 연결된 최대 컴포넌트에서 계산 (무향으로 변환)
    if N > 0:
        H = G.to_undirected()
        largest_cc_nodes = max(nx.connected_components(H), key=len)
        H_cc = H.subgraph(largest_cc_nodes).copy()
        
        if H_cc.number_of_nodes() > 1:
            L = nx.average_shortest_path_length(H_cc)
            Dia = nx.diameter(H_cc)
            print(f"Average path length L (on largest component): {L:.4f}")
            print(f"Network diameter Dia (on largest component): {Dia}")
        else:
            print("Largest component has only 1 node, cannot compute L/Dia.")
    
    # Degree / Weighted degree
    print("\n=== Degree / Weighted degree (sample) ===")
    # out-degree, in-degree (weighted)
    out_wdeg = G.out_degree(weight="weight")
    in_wdeg = G.in_degree(weight="weight")
    # 그냥 몇 개만 보기
    for n, w in list(out_wdeg)[:5]:
        print(f"{n}: weighted outdegree = {w}")
    for n, w in list(in_wdeg)[:5]:
        print(f"{n}: weighted indegree = {w}")
    
    # Betweenness centrality (unweighted shortest path 기준)
    print("\n=== Betweenness centrality (top 5) ===")
    bc = nx.betweenness_centrality(G, normalized=True, weight=None)
    for n, v in sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: BC = {v:.4f}")
    
    # Closeness centrality (무향 변환 후)
    print("\n=== Closeness centrality (top 5, undirected) ===")
    cc = nx.closeness_centrality(G.to_undirected())
    for n, v in sorted(cc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: CC = {v:.4f}")
    
    # Clustering coefficient (무향)
    print("\n=== Clustering coefficient ===")
    clustering = nx.clustering(G.to_undirected(), weight=None)
    avg_clustering = sum(clustering.values()) / len(clustering) if clustering else 0
    print(f"Average clustering coefficient: {avg_clustering:.4f}")
    
    return {
        "density": density,
        "weighted_outdegree": dict(out_wdeg),
        "weighted_indegree": dict(in_wdeg),
        "betweenness": bc,
        "closeness": cc,
        "clustering": clustering,
    }



# 3) 모듈성 Q & 커뮤니티 (Louvain)
def compute_modularity_and_communities(G):
    print("\n=== Louvain community detection & modularity Q ===")
    H = G.to_undirected()
    
    # Louvain partition: node -> community_id
    partition = community.best_partition(H, weight="weight")
    # Modularity Q
    Q = community.modularity(partition, H, weight="weight")
    print(f"Modularity Q: {Q:.4f}")
    
    # 커뮤니티 예시 출력
    communities = {}
    for node, com in partition.items():
        communities.setdefault(com, []).append(node)
    
    print(f"Number of communities: {len(communities)}")
    for cid, nodes in list(communities.items())[:3]:
        print(f"Community {cid}: {nodes[:10]}{'...' if len(nodes) > 10 else ''}")
    
    return partition, Q


# 4) HITS hub / authority
def compute_hits(G, max_iter=1000, tol=1e-08):
    print("\n=== HITS hub / authority (top 5) ===")
    hubs, authorities = nx.hits(G, max_iter=max_iter, tol=tol, normalized=True)
    
    print("Top hubs:")
    for n, v in sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: hub = {v:.4f}")
    
    print("Top authorities:")
    for n, v in sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: auth = {v:.4f}")
    
    return hubs, authorities


# 5) Network efficiency E
def network_efficiency(G):
    """
    E = 1 / [n(n-1)] * sum_{i != j} 1/d_ij
    여기서는 무향 그래프의 최단경로 기준으로 계산
    """
    H = G.to_undirected()
    nodes = list(H.nodes())
    n = len(nodes)
    if n < 2:
        return 0.0
    
    eff_sum = 0.0
    pair_count = 0
    
    for i, j in itertools.combinations(nodes, 2):
        try:
            d = nx.shortest_path_length(H, i, j)
            eff_sum += 1.0 / d
            pair_count += 1
        except nx.NetworkXNoPath:
            # 연결 안되어 있으면 그 쌍은 기여 0으로 처리
            continue
    
    if pair_count == 0:
        return 0.0
    
    # 위에서 i<j 조합만 돌렸으므로 분모는 n(n-1)/2 → 논문식에 맞추면:
    # E = (2 * eff_sum) / [n(n-1)]
    E = (2.0 * eff_sum) / (n * (n - 1))
    return E


def compute_robustness_scenarios(G, remove_sets):
    """
    remove_sets: 예) {"Scenario 1 (remove Brazil)": ["Brazil"], "Scenario 2 (remove USA)": ["USA"]}
    """
    print("\n=== Network efficiency robustness scenarios ===")
    baseline_E = network_efficiency(G)
    print(f"Baseline E: {baseline_E:.4f}")
    
    results = {"Baseline": baseline_E}
    
    for name, nodes_to_remove in remove_sets.items():
        G_copy = G.copy()
        G_copy.remove_nodes_from(nodes_to_remove)
        E_val = network_efficiency(G_copy)
        print(f"{name}: remove {nodes_to_remove} → E = {E_val:.4f}")
        results[name] = E_val
    return results



In [87]:
csv_path = "/home/user/문서/workspace/python/data/콩_무역_2023.csv"

# =========================
# 1. 네트워크 구축
# =========================
G = build_trade_network(
    csv_path,
    src_col="exporter",
    dst_col="importer",
    w_col="value",
)

# =========================
# 2. 기본 지표 계산
# =========================
measures = compute_basic_measures(G)

# 🔑 여기서 꺼내온다
out_wdeg = measures["weighted_outdegree"]
in_wdeg  = measures["weighted_indegree"]
bc       = measures["betweenness"]

# =========================
# 3. 커뮤니티 & 모듈성
# =========================
partition, Q = compute_modularity_and_communities(G)

# =========================
# 4. HITS (선택)
# =========================
hubs, authorities = compute_hits(G)

# =========================
# 5. 네트워크 효율성
# =========================
E0 = network_efficiency(G)
print(f"\nOverall network efficiency E: {E0:.4f}")

robustness_results = compute_robustness_scenarios(
    G,
    {
        "Scenario 1 (remove Brazil)": ["Brazil"],
        "Scenario 2 (remove USA)": ["USA"],
        "Scenario 3 (remove Brazil & USA)": ["Brazil", "USA"],
    },
)


=== Basic topology ===
Nodes (N): 142
Edges (M): 1000
Density ρ: 0.0499
Average path length L (on largest component): 2.3367
Network diameter Dia (on largest component): 5

=== Degree / Weighted degree (sample) ===
Brazil: weighted outdegree = 91326573.11
China, mainland: weighted outdegree = 63283.17
United States of America: weighted outdegree = 44215092.42
Paraguay: weighted outdegree = 6000665.48
Argentina: weighted outdegree = 2195566.38
Brazil: weighted indegree = 181024.33
China, mainland: weighted indegree = 103383644.16999999
United States of America: weighted indegree = 662987.19
Paraguay: weighted indegree = 9541.02
Argentina: weighted indegree = 10367695.040000001

=== Betweenness centrality (top 5) ===
United States of America: BC = 0.1589
Netherlands (Kingdom of the): BC = 0.0977
France: BC = 0.0893
Canada: BC = 0.0829
China, mainland: BC = 0.0636

=== Closeness centrality (top 5, undirected) ===
China, mainland: CC = 0.6409
United States of America: CC = 0.6380
Canada: C

In [88]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

plt.figure(figsize=(8, 6))

rng = np.random.default_rng(seed=42)

# =========================
# 1. 커뮤니티별 노드 그룹
# =========================
communities = sorted(set(partition.values()))
nodes_by_comm = {
    c: [n for n in G.nodes() if partition[n] == c]
    for c in communities
}

# =========================
# 2. 커뮤니티 중심 좌표 (원형)
# =========================
K = len(communities)
theta_c = np.linspace(0, 2*np.pi, K, endpoint=False)
R = 0.65   # 커뮤니티 중심 반경

comm_center = {
    c: (R*np.cos(theta_c[i]), R*np.sin(theta_c[i]))
    for i, c in enumerate(communities)
}

# =========================
# 3. 커뮤니티 내부 균등 분포
# =========================
pos = {}

for c, nodes_c in nodes_by_comm.items():
    n = len(nodes_c)
    if n == 0:
        continue

    theta = rng.uniform(0, 2*np.pi, n)
    r = 0.18 * np.sqrt(rng.uniform(0, 1, n))  # 서브 원 크기

    cx, cy = comm_center[c]

    for i, node in enumerate(nodes_c):
        pos[node] = (
            cx + r[i]*np.cos(theta[i]),
            cy + r[i]*np.sin(theta[i])
        )

# =========================
# 4. 노드 크기
# =========================
total_trade = {
    n: out_wdeg.get(n, 0) + in_wdeg.get(n, 0)
    for n in G.nodes()
}

node_size = [
    np.sqrt(total_trade[n]) / 12
    for n in G.nodes()
]

# =========================
# 5. 노드 색
# =========================
color_map = {c: i for i, c in enumerate(communities)}
node_color = [color_map[partition[n]] for n in G.nodes()]

# =========================
# 6. 간선 (로그 스케일 + 컷오프)
# =========================
# =========================
# 6. 간선 (최대값 기준 백분율 스케일)
# =========================
edges_draw = []
edge_width = []
edge_alpha = []

# 전체 간선 중 최대 무역량
max_w = max(G[u][v]["weight"] for u, v in G.edges())

for u, v in G.edges():
    w = G[u][v]["weight"]

    # 최대값 대비 비율 (0~1)
    ratio = w / max_w

    edges_draw.append((u, v))

    # 🔹 아주 작은 흐름: 가는 실선
    if ratio < 0.02:
        edge_width.append(0.3)
        edge_alpha.append(0.05)

    # 🔸 중간 흐름
    elif ratio < 0.1:
        edge_width.append(1.2)
        edge_alpha.append(0.25)

    # 🔶 큰 흐름
    elif ratio < 0.3:
        edge_width.append(3.0)
        edge_alpha.append(0.45)

    # 🔥 압도적 핵심 경로
    else:
        edge_width.append(7.0)
        edge_alpha.append(0.8)

nx.draw_networkx_edges(
    G,
    pos,
    edgelist=edges_draw,
    arrowstyle="->",
    arrowsize=8,
    width=edge_width,
    alpha=edge_alpha,
    edge_color="gray"
)


# =========================
# 7. 노드
# =========================
nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_size,
    node_color=node_color,
    cmap="tab20",
    edgecolors="black",
    linewidths=0.4,
    alpha=0.9
)

# =========================
# 8. 상위 국가 라벨
# =========================
top_nodes = sorted(
    total_trade,
    key=total_trade.get,
    reverse=True
)[:8]

for n in top_nodes:
    x, y = pos[n]
    plt.text(
        x, y,
        n,
        fontsize=9,
        ha="center",
        va="center",
        fontweight="bold"
    )

# =========================
# 9. 한국 강조 표시
# =========================
korea_names = [
    "Korea, Rep.",
    "South Korea",
    "Republic of Korea",
    "Korea"
]

korea_node = None
for name in korea_names:
    if name in G.nodes():
        korea_node = name
        break

if korea_node is not None:
    x, y = pos[korea_node]

    # 🔴 한국 노드 테두리 강조
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[korea_node],
        node_size=[np.sqrt(total_trade[korea_node]) / 8],  # 약간 크게
        node_color=[color_map[partition[korea_node]]],
        edgecolors="red",
        linewidths=2.0,
        alpha=1.0
    )

    # 🔴 한국 라벨
    plt.text(
        x,
        y - 0.01,
        "Korea",
        fontsize=10,
        ha="center",
        va="top",
        fontweight="bold",
        color="red"
    )
else:
    print("⚠️ 한국 노드를 찾지 못했습니다. 노드 이름을 확인하세요.")





# plt.title(
#     "Global Soybean Trade Network\n"
#     "(Uniform layout; clusters grouped; arrow width ∝ log(trade volume))",
#     fontsize=13
# )

# plt.axis("off")
# plt.tight_layout()
# plt.show()


import matplotlib as mpl

mpl.use("pgf")

mpl.rcParams.update({
    "pgf.texsystem": "xelatex",   # lualatex도 가능
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],  # 논문 기본 폰트
    "axes.unicode_minus": False,
})


plt.savefig("asset/soybean_trade_network.pgf")
plt.close()



In [89]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# 1. x, y 데이터 준비
# =========================
countries = list(G.nodes())

# x축: weighted outdegree (또는 total)
x = np.array([
    out_wdeg.get(c, 0) for c in countries
])

# y축: betweenness centrality
y = np.array([
    bc.get(c, 0) for c in countries
])

# =========================
# 2. 산점도
# =========================
plt.figure(figsize=(7, 5))

plt.scatter(
    x, y,
    s=40,
    alpha=0.6,
    color="gray"
)

# =========================
# 3. 핵심 국가 강조
# =========================
key_countries = ["Brazil", "United States of America", "China, mainland"]
key_colors = {
    "Brazil": "#1f77b4",
    "United States of America": "#d62728",
    "China, mainland": "#2ca02c",
}

for c in key_countries:
    if c in countries:
        xi = out_wdeg.get(c, 0)
        yi = bc.get(c, 0)

        plt.scatter(
            xi, yi,
            s=120,
            color=key_colors[c],
            edgecolors="black",
            zorder=3
        )

        plt.text(
            xi * 1.03,
            yi * 1.03,
            c.replace(", mainland", ""),
            fontsize=10,
            fontweight="bold"
        )

# =========================
# 3-1. 한국 강조
# =========================
korea_names = [
    "Korea",
    "Republic of Korea",
    "Korea, Rep.",
    "South Korea"
]

korea = next((k for k in korea_names if k in countries), None)

if korea is not None:
    xk = out_wdeg.get(korea, 0)
    yk = bc.get(korea, 0)

    plt.scatter(
        xk, yk,
        s=140,
        color="orange",
        edgecolors="black",
        zorder=4
    )

    plt.text(
        xk + 0.1,
        yk + 0.005,
        "Korea",
        fontsize=10,
        fontweight="bold",
        color="orange"
    )
else:
    print("⚠️ 한국 노드를 찾지 못했습니다. 노드 이름을 확인하세요.")


# =========================
# 4. 축 & 스타일
# =========================
plt.xlabel("Weighted Outdegree (Export Volume)", fontsize=11)
plt.ylabel("Betweenness Centrality", fontsize=11)

plt.title(
    "Figure 2. Volume-based vs. Path-based Importance in the Global Soybean Trade Network",
    fontsize=12
)

# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

import matplotlib as mpl

mpl.use("pgf")

mpl.rcParams.update({
    "pgf.texsystem": "xelatex",   # lualatex도 가능
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],  # 논문 기본 폰트
    "axes.unicode_minus": False,
})


plt.savefig("asset/soybean_central_outdegree.pgf")
plt.close()


In [90]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# 1. x, y 데이터 준비
# =========================
countries = list(G.nodes())

# 🔁 x축: weighted indegree (Import Volume)
x = np.array([
    in_wdeg.get(c, 0) for c in countries
])

# y축: betweenness centrality
y = np.array([
    bc.get(c, 0) for c in countries
])

# =========================
# 2. 산점도
# =========================
plt.figure(figsize=(7, 5))

plt.scatter(
    x, y,
    s=40,
    alpha=0.6,
    color="gray"
)

# =========================
# 3. 핵심 국가 강조
# =========================
key_countries = ["Brazil", "United States of America", "China, mainland"]
key_colors = {
    "Brazil": "#1f77b4",
    "United States of America": "#d62728",
    "China, mainland": "#2ca02c",
}

for c in key_countries:
    if c in countries:
        xi = in_wdeg.get(c, 0)   # 🔁 indegree
        yi = bc.get(c, 0)

        plt.scatter(
            xi, yi,
            s=120,
            color=key_colors[c],
            edgecolors="black",
            zorder=3
        )

        plt.text(
            xi * 1.03,
            yi * 1.03,
            c.replace(", mainland", ""),
            fontsize=10,
            fontweight="bold"
        )

# =========================
# 3-1. 한국 강조
# =========================
korea_names = [
    "Korea",
    "Republic of Korea",
    "Korea, Rep.",
    "South Korea"
]

korea = next((k for k in korea_names if k in countries), None)

if korea is not None:
    xk = in_wdeg.get(korea, 0)   # 🔁 indegree
    yk = bc.get(korea, 0)

    plt.scatter(
        xk, yk,
        s=140,
        color="orange",
        edgecolors="black",
        zorder=4
    )

    plt.text(
        xk * 1.03,
        yk * 1.03,
        "Korea",
        fontsize=10,
        fontweight="bold",
        color="orange"
    )
else:
    print("⚠️ 한국 노드를 찾지 못했습니다. 노드 이름을 확인하세요.")

# =========================
# 4. 축 & 스타일
# =========================
plt.xlabel("Weighted Indegree (Import Volume)", fontsize=11)
plt.ylabel("Betweenness Centrality", fontsize=11)

plt.title(
    "Figure 2. Import Dependence vs. Path-based Importance\nin the Global Soybean Trade Network",
    fontsize=12
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


/tmp/ipykernel_26110/1575768281.py:110: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
